# Smart Buildings submission (TransferPolicy)

Built from `working-code` (backlog L1-L3). Control: the model-based MPC (`code/control/policies.py`) when `best.py` is attached, otherwise the starter kit's `pid_water` baseline (vendored, Apache-2.0), behind a safety layer that clips every action into bounds, replaces NaN/inf and falls back on any exception. If the attached dataset contains `best.py` (closed-form discovered formulas), a tracker follows every room's temperature from the metadata, the observations and its own actions; a model-based controller will plan from it.

**Before submitting:** run `python code/package_submission.py [--model outputs/discovery/<run>/best.py]` in working-code, upload `dist/kaggle_dataset/` as a Kaggle Dataset, attach it here (Add Input), and check the local rehearsal: `python code/rehearse_submission.py`.

In [ ]:
import glob, os, sys

# find the attached dataset that holds our code (dist/kaggle_dataset uploaded as a Kaggle Dataset)
CANDIDATES = glob.glob("/kaggle/input/*/code/submission") + glob.glob("/kaggle/input/*/*/code/submission")
if not CANDIDATES and os.path.isdir("../dist/kaggle_dataset/code/submission"):   # running locally from working-code/kaggle
    CANDIDATES = ["../dist/kaggle_dataset/code/submission"]
assert CANDIDATES, "attach the working-code Kaggle Dataset (dist/kaggle_dataset) to this notebook"
CODE_DIR = os.path.dirname(CANDIDATES[0]); DATA_DIR = os.path.dirname(CODE_DIR)
sys.path.insert(0, CODE_DIR)
MODEL_SRC = open(os.path.join(DATA_DIR, "best.py")).read() if os.path.exists(os.path.join(DATA_DIR, "best.py")) else None
print("code:", CODE_DIR, "| model:", "best.py" if MODEL_SRC else "none (pid_water only)")

In [ ]:
from typing import Any, Dict

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server
from submission.transfer_policy import TransferPolicyCore

# Controller: the model-based MPC when the discovered model (best.py) is attached, else the kit's pid_water.
# Any failure inside the MPC falls back to pid_water's action through the safety layer.
CONTROLLER = "mpc" if MODEL_SRC else "pid_water"


def _controller(meta, spec):
    if CONTROLLER in ("mpc", "ff"):
        from control.policies import FeedForwardPolicy, MPCPolicy
        return (MPCPolicy if CONTROLLER == "mpc" else FeedForwardPolicy)(meta, spec, model_src=MODEL_SRC)
    from submission.vendor.pid_plus import PIDWaterResetPolicy
    return PIDWaterResetPolicy(meta, spec)


class TransferPolicy(smart_buildings_inference_server.Policy):
    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self.core = TransferPolicyCore(building_meta_data, action_spec, controller_cls=_controller,
                                       model_src=None if CONTROLLER in ("mpc", "ff") else MODEL_SRC)

    def get_action(self, obs: np.ndarray) -> Dict[str, Any]:
        return self.core.get_action(obs)

In [ ]:
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()